# Revision runner (draft 3)

One cell per stage of `REVISION_PLAN.md`. Use a GPU runtime. Every stage appends to
`MyDrive/revision/runs/*.csv` and skips runs that are already there, so after a
disconnect re-run the setup cells and then the interrupted stage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = '/content/Cross-Dataset-Human-Activity-Recognition'
# If the repository is private, either clone with a GitHub token or copy the repo folder
# to Drive and set REPO to that path instead.
if not os.path.isdir(REPO):
    !git clone -b claude/imu-har-orientation-robustness-4jb7bq https://github.com/minseoshinn/Cross-Dataset-Human-Activity-Recognition.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip -q install xgboost tabulate

Load the pipeline (cell 20 of `Gravity_Model.ipynb`, verbatim) and the revision code. `-n` keeps the old entry point from running.

In [ ]:
%run -i -n revision/pipeline_base.py
%run -i -n revision/revision_experiments.py

## Stage 0: audit (no training, minutes)
MotionSense sign (c33), geometry identities (c36 c37 c51), antipode (c38), heading definition (c32), cadence (c66), preprocessing and label tables (c44 c45), parameters (c42).

In [ ]:
run_stage("audit")

Time one epoch on this GPU and project the hours per stage.

In [ ]:
estimate_cost()

## Tier 1

In [ ]:
run_stage("xgb")        # CPU-bound feature extraction, cached after the first run

In [ ]:
run_stage("ablation")   # A/B/C/D on 12 pairs, 2 pretraining x 4 fine-tuning seeds

In [ ]:
run_stage("baselines")  # SO(3) aug, learned SO(3), PCA heading, Mizell, OIT

In [ ]:
run_stage("ceiling")    # in-domain ceilings on each pair label set

In [ ]:
run_stage("rotation")   # controlled device- and gravity-frame rotations, 4 datasets

In [ ]:
# UniMTS: clones the official repo, installs CLIP, downloads the checkpoint.
# Reduce cost with exp_unimts(epochs=5, max_train=20000) if needed.
run_stage("unimts")

## Tier 2

In [ ]:
run_stage("site")       # where yaw invariance must live (c39 c41)

In [ ]:
run_stage("posture")    # sit / stand / lie (c44)

In [ ]:
run_stage("hparam")     # source-only selection + target grid (c40)

In [ ]:
run_stage("inference")  # latency with CPU model, threads, batch; ONNX export (c27 c56)

## Tier 3 (optional)

In [ ]:
# run_stage("augspec"); run_stage("positions"); run_stage("labeleff"); run_stage("autoaug")

## Tables, tests, figures, headline numbers
Writes `MyDrive/revision/tables/paper_tables.md`, `T_*.csv`, `F_*.pdf`, `headline_numbers.json`.

In [ ]:
%run -i -n revision/revision_analysis.py
path = make_all_tables()
print(open(path).read()[:6000])